# Iterative Deepening Search (IDS) in Python
**Lecture 15 | Artificial Intelligence**

This notebook implements Iterative Deepening Search using the same graph as Lecture 13.

IDS repeatedly runs Depth-Limited Search (DLS), increasing the depth limit until the goal is found. Each iteration restarts from the start node.

**Implementation choices**
- The graph is represented by an adjacency list.
- DFS uses a stack (LIFO).
- Each stack entry stores `(node, depth, path)`.
- The current path prevents cycles within a branch.
- The traversal list records nodes popped during the current iteration.
- Neighbor order is preserved by pushing neighbors in reverse order.


## 1. Graph representation

The graph is undirected, so every edge appears in both endpoints' neighbor lists. Neighbor order affects traversal order.


In [ ]:
graph = {
    'A': ['B', 'D'],
    'B': ['A', 'C', 'E'],
    'C': ['B', 'E', 'F', 'G'],
    'D': ['A', 'E'],
    'E': ['B', 'C', 'D', 'G'],
    'F': ['C', 'H', 'I'],
    'G': ['C', 'E', 'I'],
    'H': ['F', 'I'],
    'I': ['F', 'G', 'H']
}

print("Adjacency list:")
for node, neighbors in graph.items():
    print(f"{node}: {neighbors}")


## 2. Required variables in IDS

- **`depth_limit`**: maximum depth allowed in the current iteration.
- **`stack`**: nodes waiting to be processed; the rightmost item is the top.
- **`path`**: route from the start to the current node; used to avoid cycles along the current branch and return the solution.
- **`traversal`**: nodes processed in the current iteration.
- **`all_traversals`**: optional list storing each iteration's traversal.
- **`start` and `goal`**: initial node and target node.

This implementation uses the current path to prevent cycles instead of a single global visited set. In IDS, a global visited set shared across depth limits can incorrectly block a state reached later with more remaining depth.


## 3. Step One — Depth-Limited Search (DLS)

DLS explores the graph only up to a specified depth limit. A node at the limit can be checked for the goal, but its neighbors are not expanded.


In [ ]:
def depth_limited_search(graph, start, goal, depth_limit):
    # Stack entries: (current node, depth, path to current node)
    stack = [(start, 0, [start])]
    traversal = []

    while stack:
        current, depth, path = stack.pop()
        traversal.append(current)

        # Goal test happens when the node is popped
        if current == goal:
            return traversal, path

        # Do not expand nodes at the depth limit
        if depth == depth_limit:
            continue

        # Push neighbors in reverse order so the first listed is processed first
        for neighbor in reversed(graph[current]):
            # Avoid cycles along this branch
            if neighbor not in path:
                stack.append((neighbor, depth + 1, path + [neighbor]))

    return traversal, None


## 4. Step Two — Iterative Deepening Search (IDS)

IDS calls DLS with limits `0, 1, 2, ...`. Every call creates a fresh stack and traversal. It stops when DLS returns a solution path.

`max_depth` is a practical safeguard; it prevents an unbounded loop if the goal is unreachable.


In [ ]:
def ids(graph, start, goal, max_depth=20, show_steps=True):
    all_traversals = []

    for depth_limit in range(max_depth + 1):
        traversal, path = depth_limited_search(
            graph, start, goal, depth_limit
        )
        all_traversals.append(traversal)

        if show_steps:
            print(f"Depth limit {depth_limit}:")
            print("  Traversal:", traversal)
            print("  Goal found:", path is not None)

        if path is not None:
            if show_steps:
                print("  Solution path:", path)
            return all_traversals, path

    if show_steps:
        print(f"Goal not found within depth limit {max_depth}.")
    return all_traversals, None


## 5. Run IDS from A to I

We use the same graph and search from `A` to `I`. Because the neighbor order is fixed, the result is reproducible.


In [ ]:
all_traversals, solution_path = ids(
    graph,
    start='A',
    goal='I',
    max_depth=20
)

print("\nFinal solution path:", solution_path)


## 6. Inspect the result by iteration

Each traversal is separate because IDS restarts the search for every new depth limit.


In [ ]:
for depth_limit, traversal in enumerate(all_traversals):
    print(f"Limit {depth_limit}: {traversal}")


## 7. Try different start and goal nodes

Change `start`, `goal`, or `max_depth`, then rerun the cell.


In [ ]:
start = 'A'
goal = 'H'

all_traversals, solution_path = ids(
    graph, start, goal, max_depth=20
)

print("\nFinal solution path:", solution_path)


## 8. What happens if the goal cannot be found within the maximum depth?

Try a maximum depth smaller than the goal's depth.


In [ ]:
all_traversals, solution_path = ids(
    graph, start='A', goal='I', max_depth=2
)

print("\nSolution path:", solution_path)


## 9. Complexity analysis

For the standard AI search-tree model:
- **Time complexity:** `O(b^d)`
- **Space complexity:** `O(bd)`

Here, `b` is the branching factor and `d` is the depth of the shallowest goal. These are standard search-tree bounds. This educational graph implementation stores a copy of the current path in each stack entry, which adds overhead compared with a memory-optimized implementation.

## 10. Practice exercises

1. Run IDS from `A` to `G`. Record the depth limit at which the goal is first found.
2. Run IDS from `D` to `H`.
3. Change the order of neighbors in the adjacency list. Compare the traversal orders and solution paths.
4. Explain why each iteration restarts from the start node.
5. Explain why a single global visited set shared across iterations can be incorrect.
6. Set `max_depth=1` for goal `I`. What does the function return?

### Key takeaway

IDS repeatedly applies DLS with increasing depth limits. The stack controls depth-first exploration, the current path prevents cycles along a branch, and the traversal list records processed nodes for each iteration.
